# Cross-Border KYC Agent — Amazon Bedrock + LangGraph + OpenRegistry MCP

Walk a company's beneficial-ownership chain across **27 national company registries** in a single conversation, using Amazon Bedrock (Anthropic Claude) as the planner and the [OpenRegistry](https://openregistry.sophymarine.com) hosted MCP server as the data source.

OpenRegistry is a free, hosted Streamable-HTTP MCP server that proxies UK Companies House, Germany Handelsregister, France Sirene+RNE, Italy InfoCamere via EU BRIS, Spain BORME, Korea OPENDART, plus 21 more — every tool call is a real-time query against the upstream government API and the response is returned **unmodified** (every field name, status string, and raw filing byte preserved verbatim).

**By the end of this notebook you'll be able to:**

- Connect Bedrock-Claude to a remote Streamable-HTTP MCP server using `langchain-mcp-adapters`.
- Build a LangGraph ReAct agent that auto-discovers ~30 OpenRegistry tools at runtime.
- Walk a UK company's PSC chain across borders, with every fact cited back to the upstream government register.
- Handle CJEU C-37/20 access-restricted UBO registers (DE / ES / IT / NL / LU / AT / MT / PT) honestly, by surfacing the `alternative_url` instead of substituting aggregator data.

## 1. Installation

In [ ]:
%%capture
%pip install -U langchain-aws langchain-mcp-adapters langgraph mcp boto3 nest_asyncio python-dotenv

## 2. Configuration

We use Anthropic Claude Sonnet 4.5 on Bedrock by default, in `us-west-2`. OpenRegistry's hosted endpoint is `https://openregistry.sophymarine.com/mcp` — anonymous tier, no API key required.

In [ ]:
import os
import nest_asyncio

nest_asyncio.apply()

BEDROCK_REGION = os.environ.get("AWS_REGION", "us-west-2")
BEDROCK_MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

OPENREGISTRY_MCP_URL = os.environ.get(
    "OPENREGISTRY_MCP_URL", "https://openregistry.sophymarine.com/mcp"
)
OPENREGISTRY_TOKEN = os.environ.get("OPENREGISTRY_TOKEN")  # optional, only for paid tiers

## 3. Initialize the Bedrock LLM

We use `ChatBedrockConverse` from `langchain-aws` so the model can stream tool calls through LangGraph. The `temperature=0` keeps the agent's tool selection deterministic on repeat runs.

In [ ]:
from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model=BEDROCK_MODEL_ID,
    region_name=BEDROCK_REGION,
    temperature=0,
    max_tokens=4096,
)
print(f"LLM ready: {BEDROCK_MODEL_ID} in {BEDROCK_REGION}")

## 4. Connect to the OpenRegistry MCP server

We use `MultiServerMCPClient` from `langchain-mcp-adapters` to attach the remote server. The client opens a Streamable-HTTP session, calls MCP `tools/list`, and converts every advertised tool into a LangChain `BaseTool` instance — so Claude on Bedrock sees ~30 first-class tools at function-calling time.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

connection = {
    "url": OPENREGISTRY_MCP_URL,
    "transport": "streamable_http",
}
if OPENREGISTRY_TOKEN:
    connection["headers"] = {"Authorization": f"Bearer {OPENREGISTRY_TOKEN}"}

mcp_client = MultiServerMCPClient({"openregistry": connection})
tools = await mcp_client.get_tools()
print(f"Discovered {len(tools)} tools from OpenRegistry:")
for t in tools[:8]:
    print(f"  - {t.name}")
if len(tools) > 8:
    print(f"  ... +{len(tools) - 8} more")

## 5. Build the LangGraph ReAct agent

The agent has a single role: walk corporate-ownership chains across borders. The system prompt teaches Claude on Bedrock to:

1. Identify the relevant jurisdiction from any company name (`'gb'` for UK, `'de'` for Germany, etc.).
2. Quote the registry's own `nature_of_control` strings verbatim — never normalise them.
3. Recurse into corporate PSCs by jumping to their home jurisdiction.
4. Surface `alternative_url` when an upstream register returns HTTP 501 (CJEU C-37/20 access-gated).

In [ ]:
from langgraph.prebuilt import create_react_agent

SYSTEM_PROMPT = (
    "You are a cross-border KYC / due-diligence assistant with live access to 27 "
    "national company registries via the OpenRegistry MCP server. When asked "
    "about a company, identify the relevant jurisdiction (ISO 3166-1 alpha-2, "
    "e.g. 'gb' for UK, 'de' for Germany, 'fr' for France) and use the OpenRegistry "
    "tools to look it up. Always pass jurisdiction='<code>' explicitly. Quote the "
    "registry's own field names verbatim — do not normalise PSC `nature_of_control` "
    "values. When walking corporate ownership chains across borders, recurse "
    "jurisdiction by jurisdiction until you reach an individual or hit an "
    "AML-gated register. If a tool returns HTTP 501 with an `alternative_url`, "
    "surface that URL to the user — that signals a CJEU C-37/20-restricted "
    "register (DE / ES / IT / NL / LU / AT / MT / PT) which only AML-obliged "
    "entities can query. Always cite the registry and the company identifier you "
    "looked up so the user can verify against the government source."
)

agent = create_react_agent(
    llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
)
print("Agent ready. Tools attached:", len(tools))

## 6. Run the agent — walk Revolut Ltd's PSC chain

We pick **Revolut Ltd** (UK Companies House `08804411`) as the seed entity. The agent should:

1. Search Companies House for the entity (or accept the company number directly).
2. Call `get_persons_with_significant_control` on it.
3. For each corporate PSC, jump to its jurisdiction and repeat.
4. Stop when the chain reaches an individual or hits an AML-gated register.
5. Cite every hop.

On a typical run you should see 5–10 live MCP tool calls against UK Companies House and any cross-border parents the chain reaches.

In [ ]:
USER_PROMPT = (
    "Walk Revolut Ltd's PSC chain (UK Companies House company number 08804411) "
    "across jurisdictions until you reach an individual or hit an AML-gated "
    "register. Cite the registry and identifier for each hop, and quote the "
    "upstream `nature_of_control` strings verbatim."
)

result = await agent.ainvoke({"messages": [("user", USER_PROMPT)]})

for m in result["messages"]:
    role = m.type if hasattr(m, "type") else m.__class__.__name__
    content = m.content if hasattr(m, "content") else ""
    if hasattr(m, "tool_calls") and m.tool_calls:
        for tc in m.tool_calls:
            print(f"  → tool: {tc['name']}({tc['args']})")
    if content:
        text = content if isinstance(content, str) else str(content)
        print(f"[{role}] {text[:600]}{'...' if len(text) > 600 else ''}\n")

## 7. Try other prompts

Some patterns worth trying with the same agent:

- *"Find Tesco PLC on UK Companies House (jurisdiction `gb`) and pull the three most recent filings — quote the document_id values verbatim."*
- *"Pull Samsung Electronics' latest annual disclosure from Korean OpenDART — list the major shareholders disclosed in the most recent filing."*
- *"Search Spanish BORME for any actos inscritos mentioning Inditex SA in the last 90 days."*
- *"For an EU/UK Ltd of your choice, walk the corporate-ownership chain across jurisdictions and identify which (if any) layers cross into a CJEU C-37/20 restricted register — call out which AML channel would be needed to continue."*

## 8. Production considerations

**Rate limits.** Anonymous tier is 20 req/min/IP with a 3-country fan-out cap. For batch onboarding, complete the OAuth 2.1 flow at [openregistry.sophymarine.com/account](https://openregistry.sophymarine.com/account) and pass the bearer token via the `OPENREGISTRY_TOKEN` env var.

**Provenance.** Every OpenRegistry response keeps the registry's own `company_id` so any fact in the agent's answer can be reconstructed back to the government URL. The Enterprise tier pre-synthesises `source_url` / `registry_url` / `data_license` into every response for one-click audit-trail in compliance reports.

**AML gates.** Eight EU jurisdictions had their UBO registers gated to AML-obliged entities only after CJEU C-37/20 (Nov 2022). The agent handles this honestly — when the upstream returns HTTP 501, the answer surfaces `alternative_url` so the operator knows which AML-obliged channel to use, instead of silently substituting commercial-aggregator data.

## See also

- [OpenRegistry docs / Bedrock integration page](https://openregistry.sophymarine.com/docs/integrations/aws-bedrock)
- 40 Claude Agent Skills shipping the same workflows: <https://github.com/sophymarine/openregistry/tree/main/skills>
- Live capability matrix per jurisdiction: call the agent's `list_jurisdictions` tool.